In [ ]:
import pandas as pd

In [5]:
from sklearn.metrics import roc_curve, auc
import scorecardpy as sc
import numpy as np

# scorecrad for tree
1. 适用于树模型的评分卡函数，计算每个特征的分箱得分
2. 通过特征重要性和分箱值计算每个特征的分箱得分
3. 计算样本各特征的得分，得到总分

In [ ]:
def scorecard_for_tree(bins, model, xcolumns, points0=600, odds0=1/19, pdo=50, basepoints_eq0=False, digits=0):
    """
    适用于树模型的评分卡函数
    
    Parameters:
    -----------
    model : 树模型（如LightGBM、XGBoost模型）
        需要有feature_importances_属性
    bins : dict
        每个特征的分箱值字典
    xcolumns : list
        特征列名列表
    points0 : float, default=600
        基础分数
    odds0 : float, default=1/19
        基础赔率,表示在基准分数下，好客户与坏客户的比例为 20:1
    pdo : float, default=50
        点数倍率, 即odds翻倍时的分数变化，表示每增加 50 分，好客户的比例翻倍
    basepoints_eq0 : bool, default=False
        是否将基础分设为0
    digits : int, default=0
        分数的小数位数
    
    Returns:
    --------
    scores : dict
        特征分箱得分字典，每个特征的分箱得分为一个字典，键为分箱值，值为分数
    """
    
    
    # 获取特征重要性
    importances = model.feature_importances_
    
    # 将特征重要性归一化，使其和为1
    normalized_importances = importances / np.sum(importances)
    
    # 计算基础分和因子
    factor = pdo / np.log(2)   
    offset = points0 - factor * np.log(odds0)
    
    # 计算每个特征的得分
    scores = {}
    total_bins = sum(len(bins[col]) for col in xcolumns)
    
    for i, col in enumerate(xcolumns):
        # 计算该特征的总分配分数
        feature_score = normalized_importances[i] * (points0 if basepoints_eq0 else (points0 - offset))
        # 将分数平均分配给每个分箱
        bin_scores = np.linspace(-feature_score/2, feature_score/2, len(bins[col]))  # 等差数列
        scores[col] = {bin_val: round(score, digits) 
                      for bin_val, score in zip(bins[col], bin_scores)}
    
    # 如果不将基础分设为0，则添加offset到总分中
    if not basepoints_eq0:
        scores['offset'] = round(offset, digits)
    
    return scores

In [ ]:
def calculate_feature_directions_and_weights(model, X, feature_names, num_points=10):
    """
    计算特征方向和每个分箱的权重
    """
    feature_info = {}
    
    for feature in feature_names:
        # 计算部分依赖
        pdp_results = partial_dependence(model, X, [feature], kind="average", grid_resolution=num_points)
        
        # 提取特征值和对应的部分依赖值
        feature_values = pdp_results["values"][0]
        pdp_values = pdp_results["average"][0]
        
        # 归一化PDP值到[-1, 1]范围
        pdp_normalized = (pdp_values - np.min(pdp_values)) / (np.max(pdp_values) - np.min(pdp_values)) * 2 - 1
        
        # 计算相关系数
        correlation = np.corrcoef(feature_values, pdp_values)[0, 1]
        
        feature_info[feature] = {
            'direction': 1 if correlation > 0 else -1,
            'pdp_values': pdp_normalized,
            'correlation': abs(correlation)  # 相关系数的绝对值
        }
        
        # 可选：绘制部分依赖图
        plt.figure(figsize=(10, 6))
        plt.plot(feature_values, pdp_values)
        plt.title(f'Partial Dependence Plot for {feature} (corr: {correlation:.3f})')
        plt.xlabel(feature)
        plt.ylabel('Partial Dependence')
        plt.savefig(f'pdp_{feature}.png')
        plt.close()

    return feature_info

def assign_bin_scores(feature_score, bins_count, pdp_values, correlation_strength):
    """
    根据PDP值和相关系数强度分配分数
    """
    # 使用相关系数强度来调整分数分配
    weights = pdp_values * correlation_strength
    
    # 确保权重和为0，这样特征的总体得分不变
    weights = weights - np.mean(weights)
    
    # 根据feature_score调整权重的尺度
    bin_scores = weights * feature_score
    
    # 如果分箱数量与PDP值数量不同，进行插值
    if bins_count != len(bin_scores):
        old_indices = np.linspace(0, 1, len(bin_scores))
        new_indices = np.linspace(0, 1, bins_count)
        bin_scores = np.interp(new_indices, old_indices, bin_scores)
    
    return bin_scores

def scorecard_for_tree(bins, model, X, xcolumns, points0=600, odds0=1/19, pdo=50, basepoints_eq0=False, digits=0):
    # 获取特征重要性
    importances = model.feature_importances_
    normalized_importances = importances / np.sum(importances)
    
    # 计算基础分和因子
    factor = pdo / np.log(2)
    offset = points0 - factor * np.log(odds0)
    
    # 计算特征方向和PDP值
    feature_info = calculate_feature_directions_and_weights(model, X, xcolumns)
    
    # 计算每个特征的得分
    scores = {}
    for i, col in enumerate(xcolumns):
        # 计算该特征的总分配分数
        feature_score = normalized_importances[i] * (points0 if basepoints_eq0 else (points0 - offset))
        
        # 根据PDP值和相关系数强度分配分数
        bin_scores = assign_bin_scores(
            feature_score=feature_score,
            bins_count=len(bins[col]),
            pdp_values=feature_info[col]['pdp_values'],
            correlation_strength=feature_info[col]['correlation']
        )
        
        # 如果是负相关，翻转分数
        if feature_info[col]['direction'] == -1:
            bin_scores = -bin_scores
            
        scores[col] = {bin_val: round(score, digits) 
                      for bin_val, score in zip(bins[col], bin_scores)}
    
    # 如果不将基础分设为0，则添加offset到总分中
    if not basepoints_eq0:
        scores['offset'] = round(offset, digits)
    
    return scores


In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

# 准备示例数据
X = pd.DataFrame({
    'age': [25, 35, 45, 55],
    'income': [30000, 50000, 70000, 90000],
    'credit_score': [600, 650, 700, 750]
})
y = [0, 1, 0, 1]

# 训练模型
model = lgb.LGBMClassifier()
model.fit(X, y)

# 定义分箱
bins = {
    'age': [20, 30, 40, 50, 60],
    'income': [20000, 40000, 60000, 80000, 100000],
    'credit_score': [550, 600, 650, 700, 750]
}

# 计算评分卡
scores = scorecard_for_tree(
    bins=bins,
    model=model,
    X=X,
    xcolumns=['age', 'income', 'debt_ratio'],
    points0=600,
    odds0=1/19,
    pdo=50,
    basepoints_eq0=False,
    digits=0
)


# 使用评分卡计算总分
def calculate_total_score(sample, scores):
    total = scores.get('offset', 0)  # 获取基础分
    for col in sample.keys():
        if col in scores:
            # 找到最接近的分箱值
            bin_values = list(scores[col].keys())
            closest_bin = min(bin_values, key=lambda x: abs(x - sample[col])) # 找到最接近的分箱值
            total += scores[col][closest_bin]
    return total

# 示例使用
sample = {
    'age': 32,
    'income': 45000,
    'credit_score': 680
}
total_score = calculate_total_score(sample, scores)
print(f"Total score: {total_score}")

In [2]:
!python -m ipykernel --version
#7.16.1import sys
print(sys.path)

8.31.0
['/Applications/PyCharm.app/Contents/plugins/python/helpers-pro/jupyter_debug', '/Applications/PyCharm.app/Contents/plugins/python/helpers/pydev', '/opt/anaconda3/envs/job310/lib/python310.zip', '/opt/anaconda3/envs/job310/lib/python3.10', '/opt/anaconda3/envs/job310/lib/python3.10/lib-dynload', '', '/opt/anaconda3/envs/job310/lib/python3.10/site-packages', '/opt/anaconda3/envs/job310/lib/python3.10/site-packages/setuptools/_vendor']
